In [2]:
import torch
import torch.nn as nn
print(torch.__version__)

2.7.1+cu126


## **SECTION C.0: PREPARE DATA**

When working with tokenization, punctuation is often attached to words.

For example, in: *"Hello, world!"*
the comma and the exclamation mark are stuck to the words.
A common preprocessing step is to separate punctuation from words so that each punctuation mark becomes its own token.
#### Forward Process: Split Text with `re.split`

Example:

```python
import re

text = "Hello, world!"
tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
print(tokens)
```
#### What **`re.split`** Does

Syntax:

```python
re.split(pattern, string)
```
It means:

- search the string for everything that matches `pattern`
- split the string at each match
If the regex contains a **capturing group** `(...)`, the matched separators are also kept in the result.
#### Meaning of the Regex
```python
r'([,.:;?_!"()\']|--|\s)'
```
- `r'...'`  
  raw string for regex

- `(...)`  
  capturing group  
  This means the matched separator will also appear in the result

- `[,.:;?_!"()\']`  
  match one punctuation character from this list:
  - `,`
  - `.`
  - `:`
  - `;`
  - `?`
  - `_`
  - `!`
  - `"`
  - `(`
  - `)`
  - `'`

- `--`  
  match the two-character string `"--"`

- `\s`  
  match any whitespace character:
  - space
  - tab
  - newline

- `|`  
  means **or**

So the regex says:

> split whenever you find:
> - one punctuation mark from the list
> - or `--`
> - or whitespace
#### Example and Meaning

Input:

```python
"Hello, world!"
```
Result of:

```python
re.split(r'([,.:;?_!"()\']|--|\s)', text)
```
is something like:

```python
['Hello', ',', '', ' ', 'world', '!', '']
```
Usually, we remove empty strings and spaces:
```python
tokens = [x for x in tokens if x.strip()]
print(tokens)
```
Output:

```python
['Hello', ',', 'world', '!']
```
# Reverse Process: Fix Text Back with `re.sub`

After splitting text into tokens, punctuation is often separated from words by spaces.
For example, if we join tokens with spaces:
```python
['Hello', ',', 'world', '!']
```
we may get:

```python
"Hello , world !"
```

But this is not natural writing.
We want to rebuild the correct sentence:
```python
"Hello, world!"
```
To do that, we use `re.sub`.

#### What `re.sub` Does
Syntax:

```python
re.sub(pattern, replacement, string)
```

It means:

- search the string for everything that matches `pattern`
- replace each match with `replacement`
#### Example

```python
import re
text = "Hello , world !"
fixed = re.sub(r'\s+([,.:;?!])', r'\1', text)
print(fixed)
```
Output:

```python
Hello, world!
```

---

#### Meaning of the Regex

```python
r'\s+([,.:;?!])'
```

##### Breakdown

- `r'...'`  
  raw string for regex
- `\s+`  
  match one or more whitespace characters

- `([,.:;?!])`  
  capture one punctuation mark from this list:
  - `,`
  - `.`
  - `:`
  - `;`
  - `?`
  - `!`

So the regex says:

> find spaces followed by punctuation

Examples of matches:
- `" ,"`
- `" !"`
- `" ?"`
## Meaning of the Replacement
```python
r'\1'
```
This means:
- replace the whole match with the content of the **first captured group**
The first captured group is:
```python
([,.:;?!])
```
So if the match is:
```python
" ,"
```
then `\1` is:
```python
","
```
That means:
- the spaces are removed
- the punctuation is kept
####  Example

Start with:
```python
"Hello , world !"
```
The regex finds:
- `" ,"`
- `" !"`
Then each one is replaced by:

- `","`
- `"!"`
So the final result becomes:

```python
"Hello, world!"
```
#### **Other Useful Reverse Fixes**

##### Remove spaces after an opening parenthesis

```python
re.sub(r'(\()\s+', r'\1', text)
```
- `(\()` captures `"("`
- `\s+` matches spaces after it
So it says:
> find an opening parenthesis followed by spaces, and keep only the parenthesis

Example:

```python
"( hello"
```
becomes:
```python
"(hello"
```
##### Remove spaces before a closing parenthesis

```python
re.sub(r'\s+(\))', r'\1', text)
```

##### Meaning

- `\s+` matches spaces
- `(\))` captures `")"`

So it says:

> find spaces before a closing parenthesis, and keep only the closing parenthesis

Example:
```python
"hello )"
```
becomes:
```python
"hello)"
```


In [3]:
import os 

file_path="the-verdict.txt"
with open(file_path, 'r', encoding="utf-8") as f :
    raw_text=f.read()


print("total number of character", len(raw_text))
print(raw_text[:99]) 


total number of character 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [4]:
print(raw_text[:30])

I HAD always thought Jack Gisb


In [5]:
import re
preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])
all_words=sorted(set(preprocessed))
vocab_size=len(all_words)
vocab={
    token : integer for integer,token in enumerate(all_words)
}

print(vocab_size)
print(len(preprocessed))

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
1130
4690


In [6]:
text="hello! how are you 'sylvain'?]"
import re 
preprocessed=re.split(r'([#{|`!\^_()}\]\']|--|\s)', text)
print(preprocessed)

['hello', '!', '', ' ', 'how', ' ', 'are', ' ', 'you', ' ', '', "'", 'sylvain', "'", '?', ']', '']


# **TOKENSIZER**

In [7]:
import re
class SimpleTokenizer() : 
    def __init__(self, vocab, special_words) :
        
        self.str_to_int=vocab
        self.vocab_size=len(vocab)
        for word in special_words : 
            vocab[word]=self.vocab_size
            self.vocab_size+=1
        self.int_to_str= {i:s for s,i in vocab.items()}


    def encode(self, text) : 
        preprocessed= re.split(r'([,;:!?./§^\(\)\[\]"`]|--|\s)' , text)
        preprocessed=[x for x in preprocessed if x.strip()]
        preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        ids=[self.str_to_int[item] for item in preprocessed]

        return ids
    
    def decode(self, ids) : 
        text_list=[self.int_to_str[id] for id in ids]
        raw_text=" ".join(text_list)
        text=re.sub(r'(\s+(,;:!?./§^\(\)\[\]"`))' , r'\1' , raw_text)
        return text 


In [8]:
## TEST THE TOKENIZER 
special_words=["<|unk|>", "<|endoftext|>"]
Tokenizer=SimpleTokenizer(vocab, special_words)
text="i would like to be fluent in english"

token_ids=Tokenizer.encode(text)
print(token_ids)

#recover the text
recover_text=Tokenizer.decode(token_ids)
print(recover_text)

[1130, 1120, 628, 1016, 198, 1130, 568, 1130]
<|unk|> would like to be <|unk|> in <|unk|>


In [9]:
# import download_file
from pathlib import Path

class Qwen3Tokenizer:
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
    ]
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>)")

    def __init__(self, tokenizer_file_path="tokenizer-base.json",
                 apply_chat_template=False,
                 add_generation_prompt=False,
                 add_thinking=False):
        from tokenizers import Tokenizer

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        tok_path = Path(tokenizer_file_path)
        if not tok_path.is_file():
            raise FileNotFoundError(
                f"Tokenizer file '{tok_path}' not found. Please ensure it's available."
            )

        self._tok = Tokenizer.from_file(str(tok_path))
        self._special_to_id = {t: self._tok.token_to_id(t) for t in self._SPECIALS}

        self.pad_token = "<|endoftext|>"
        self.pad_token_id = self._special_to_id.get(self.pad_token)

        # Match HF behavior: chat model → <|im_end|>, base model → <|endoftext|>
        fname = tok_path.name.lower()
        if "base" in fname and "reasoning" not in fname:
            self.eos_token = "<|endoftext|>"
        else:
            self.eos_token = "<|im_end|>"
        self.eos_token_id = self._special_to_id.get(self.eos_token)

    def encode(self, prompt, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        stripped = prompt.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        if chat_wrapped:
            prompt = self._wrap_chat(prompt)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(prompt)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    def decode(self, token_ids):
        return self._tok.decode(token_ids, skip_special_tokens=False)

    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"  # insert no <think> tag, just a new line
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s


class KVCache:
    def __init__(self, n_layers):
        self.cache = [None] * n_layers

    def get(self, layer_idx):
        return self.cache[layer_idx]

    def update(self, layer_idx, value):
        self.cache[layer_idx] = value

    def get_all(self):
        return self.cache

    def reset(self):
        for i in range(len(self.cache)):
            self.cache[i] = None


# def download_qwen3_small(kind="base", tokenizer_only=False, out_dir="."):
#     files = {
#         "base": {"model": "qwen3-0.6B-base.pth", "tokenizer": "tokenizer-base.json"},
#         "reasoning": {"model": "qwen3-0.6B-reasoning.pth", "tokenizer": "tokenizer-reasoning.json"},
#     }
#     if kind not in files:
#         raise ValueError("kind must be 'base' or 'reasoning'")

#     repo = "rasbt/qwen3-from-scratch"
#     hf_fmt = "https://huggingface.co/{repo}/resolve/main/{file}"
#     backup_root = "https://f001.backblazeb2.com/file/reasoning-from-scratch/qwen3-0.6B"
#     targets = ["tokenizer"] if tokenizer_only else ["model", "tokenizer"]

#     for key in targets:
#         fname = files[kind][key]
#         primary = hf_fmt.format(repo=repo, file=fname)
#         backup = f"{backup_root}/{fname}"
#         download_file(primary, out_dir=out_dir, backup_url=backup)


def load_hf_weights_into_qwen(model, param_config, params):
    """
    Only used in Appendix D for loading the other Qwen3 variants.
    """
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    for l in range(param_config["n_layers"]):  # noqa: E741
        block = model.trf_blocks[l]
        att = block.att

        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # QK norms
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # Attention layernorm
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # Feedforward weights
        if "num_experts" in param_config:
            # Load router (gating) weights
            block.ff.gate.weight = assign(
                block.ff.gate.weight,
                params[f"model.layers.{l}.mlp.gate.weight"],
                f"model.layers.{l}.mlp.gate.weight"
            )
            # Load expert weights
            for e in range(param_config["num_experts"]):
                prefix = f"model.layers.{l}.mlp.experts.{e}"
                block.ff.fc1[e].weight = assign(
                    block.ff.fc1[e].weight,
                    params[f"{prefix}.gate_proj.weight"],
                    f"{prefix}.gate_proj.weight"
                )
                block.ff.fc2[e].weight = assign(
                    block.ff.fc2[e].weight,
                    params[f"{prefix}.up_proj.weight"],
                    f"{prefix}.up_proj.weight"
                )
                block.ff.fc3[e].weight = assign(
                    block.ff.fc3[e].weight,
                    params[f"{prefix}.down_proj.weight"],
                    f"{prefix}.down_proj.weight"
                )
                # After assigning weights, move the expert layers from meta to CPU
                block.ff.fc1[e] = block.ff.fc1[e].to("cpu")
                block.ff.fc2[e] = block.ff.fc2[e].to("cpu")
                block.ff.fc3[e] = block.ff.fc3[e].to("cpu")

        else:
            block.ff.fc1.weight = assign(
                block.ff.fc1.weight,
                params[f"model.layers.{l}.mlp.gate_proj.weight"],
                f"model.layers.{l}.mlp.gate_proj.weight"
            )
            block.ff.fc2.weight = assign(
                block.ff.fc2.weight,
                params[f"model.layers.{l}.mlp.up_proj.weight"],
                f"model.layers.{l}.mlp.up_proj.weight"
            )
            block.ff.fc3.weight = assign(
                block.ff.fc3.weight,
                params[f"model.layers.{l}.mlp.down_proj.weight"],
                f"model.layers.{l}.mlp.down_proj.weight"
            )

        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # Final normalization and output head
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")

## SECTION C.1 : ROOT MEANS SCARE LAYER NORMALIZATION (RMSNorm)
* This componnent is used in replacement of the **LayerNorm** used in the GPT-2 model and is increasily common in recent model architectures.
* The rule is the same as the **LayerNorm** which is normalizing the layer activations to stabilize and improve the training. 
* This component is different from the LayerNorm by the fact that we didn't include the mean-centering step.
* The activations will still normalized, but they are not centered at 0.
* During the implementation, take care of : 
  * Use the input_type of the activation with : *x.dtype*
  * Use torch.float32 for a better precision of the operation.
  * Compute the mean of the variance in the last dimension of the activation which shape should probably be (batch_size , seq_len, emb_dim)
  * Use torch.rsqrt(x) to compute the reciprocal root scare $\frac{1}{\sqrt{x}}$
  * Make sure to bring back the output into the same type of the activation input to avoid errors
  * $$RMSNorm(x)=\frac{x}{\sqrt{\frac{1}{D}\sum_{i=1}^{D}x_{i}^2}} . \gamma$$
<img src="images/rmsnorm.png" alt="RMSNorm formula" >



In [10]:
class RMSNorm(nn.Module) : 
    def __init__(self,
            emb_dim, 
            eps=1e-6, 
            bias=False, 
    ) : 
        super().__init__()
        self.scale=nn.Parameter(torch.ones(emb_dim))
        self.eps=eps
        self.shift = None if not bias else nn.Parameter(torch.zeros(emb_dim))


    def forward(self , x) : 
        # x shape= (Batch_size, len_seq,emb_dim)
        input_type=x.dtype
        x=x.to(torch.float32)

        variance=x.pow(2).mean(dim=-1, keepdim=True)
        norm_x=x*torch.rsqrt(variance+self.eps)*self.scale    #use the torch.rsqrt which means reciprocal root scare
        if self.shift is not None : 
            norm_x=norm_x+self.shift  
        return norm_x.to(input_type) #important to make sure to return the same type as input

## **SECTION 2** : GLU / SwiGLU intuition in the feed-forward block

A GLU-like feed-forward block can be understood as a **multiplicative feature modulation** mechanism.

The input $x$ is projected into two different vectors:

$
a = W_v x, \qquad g = W_g x
$

- $a$: **candidate feature vector: represent what to say**
- $g$: **gating signal : represent what to let passed**

In **GLU**, the output is:

$
\mathrm{GLU}(x) = a \odot \sigma(g)
$

where $\odot$ is the elementwise product.

For each coordinate $i$,

$
y_i = a_i \, \sigma(g_i)
$

Since $\sigma(g_i) \in (0,1)$, the term $\sigma(g_i)$ acts like a **soft coefficient** that controls how much of the candidate feature $a_i$ is kept. That is why we can think of it as a **Gate**.

So the most literal interpretation is:

- $a_i$: proposed feature value
- $\sigma(g_i)$: how strongly this feature should be kept

This is why GLU can be viewed as a form of **multiplicative feature selection or modulation**.



In **SwiGLU**, the sigmoid gate is replaced by a SiLU-transformed branch:

$
\mathrm{SwiGLU}(x) = \mathrm{SiLU}(W_g x) \odot (W_v x)
$

with

$
\mathrm{SiLU}(z) = z \sigma(z)
$

In this case, the “gate” is no longer restricted to $(0,1)$. It can be:

- close to $0$,
- positive and larger than $1$,
- slightly negative.

So in SwiGLU, the word **gate** is only approximate. It is more accurate to say that the SiLU branch provides a **smooth multiplicative modulation** of the candidate features.

Therefore:

- **GLU** = candidate features $\times$ soft gate
- **SwiGLU** = candidate features $\times$ smooth learned modulation

This is the main intuition behind using GLU-like mechanisms inside the transformer feed-forward block.

Given an input tensor

$$
x \in \mathbb{R}^{B \times T \times D}
$$

the GLU / SwiGLU transformation produces a hidden representation

$$
h \in \mathbb{R}^{B \times T \times D_{ff}}
$$

where:
- $B$ is the batch size
- $T$ is the sequence length
- $D$ is the model dimension
- $D_{ff}$ is the feed-forward hidden dimension

The full feed-forward block then projects this hidden representation back to the model dimension, producing

$$
y \in \mathbb{R}^{B \times T \times D}
$$

### Intuition

* The block takes token representations as input and processes each token independently.  
* It first maps each token vector into two higher-dimensional vectors.  
* One branch acts as a gate, and the other carries the values.  
* These two branches are combined through elementwise multiplication, which produces a modulated hidden representation:

$$
h = \sigma(W_g x) \odot (W_v x)
$$

In the full FFN block, this hidden representation is then projected back to the original model dimension.

<img src="images/ffn.png" alt="RMSNorm formula">


In [11]:
class FeedForward(nn.Module) : 

    def __init__(self, cfg) : 
        super().__init__()
        self.fc1=nn.Linear(in_features=cfg["emb_dim"], out_features=cfg["hidden_dim"], bias=False ,dtype=cfg["dtype"])
        self.fc3=nn.Linear(in_features=cfg["hidden_dim"], out_features=cfg["emb_dim"] , bias=False ,dtype=cfg["dtype"])
        self.fc2=nn.Linear(in_features=cfg["emb_dim"] , out_features=cfg["hidden_dim"] , bias=False ,dtype=cfg["dtype"])
        self.dtype=cfg["dtype"]

    def forward(self, x): 
        #x shape=(batch_size ,seq_len, emb_dim)
        # x=x.to(self.dtype)

        gate=nn.functional.silu(self.fc1(x)) #(B,T,hidden_dim)
        content=self.fc2(x)
        output=self.fc3(gate*content)

        return output

In [12]:
batch_size=2
seq_len=4
emb_dim=16
hidden_dim=4
dtype=torch.float16
cfg={
    "emb_dim":emb_dim,
    "hidden_dim" : hidden_dim,
    "dtype" : dtype , 


}
A=torch.randn(batch_size,seq_len,emb_dim, dtype=dtype)
ffn=FeedForward(cfg)
print(ffn)
print(A)
# print(ffn(A))
print(ffn(A).shape)

FeedForward(
  (fc1): Linear(in_features=16, out_features=4, bias=False)
  (fc3): Linear(in_features=4, out_features=16, bias=False)
  (fc2): Linear(in_features=16, out_features=4, bias=False)
)
tensor([[[ 1.3291,  2.4141,  0.7861,  0.5488,  0.2006, -0.7261,  1.2930,
           0.6963, -0.2119,  0.9478,  0.7534, -0.2817,  0.2942,  1.2998,
           2.1602, -0.0068],
         [-0.7749,  0.0039, -1.4131, -0.0807,  1.3965, -0.3018, -1.0576,
          -0.8931,  1.9541, -0.2769,  0.5996, -0.1702,  0.0643,  1.0898,
           1.8008, -0.7891],
         [ 1.2793, -0.1360, -0.6079, -0.9946, -0.8428, -0.8628,  0.2659,
           0.3767, -1.1768,  1.8398, -0.0673,  0.0087, -0.7686,  1.4375,
           0.3872, -0.5151],
         [-0.9521,  0.3936,  2.0469,  1.4561, -0.7544,  0.9775,  1.8252,
           0.5396, -0.7144, -0.7031, -0.7373, -1.5801, -1.8672, -0.0134,
           0.4929, -0.6040]],

        [[ 0.6328,  1.2227,  0.8447,  0.5703, -0.6226, -0.9404, -0.2629,
          -2.2461,  1.2754, -1

In [13]:
rmsnorm=RMSNorm(cfg['emb_dim'])
print(rmsnorm(A))

tensor([[[ 1.2109,  2.1992,  0.7163,  0.5000,  0.1827, -0.6616,  1.1777,
           0.6343, -0.1931,  0.8638,  0.6865, -0.2568,  0.2681,  1.1846,
           1.9688, -0.0062],
         [-0.7773,  0.0039, -1.4170, -0.0809,  1.4004, -0.3027, -1.0605,
          -0.8960,  1.9600, -0.2776,  0.6016, -0.1707,  0.0645,  1.0928,
           1.8066, -0.7915],
         [ 1.4453, -0.1537, -0.6870, -1.1240, -0.9526, -0.9751,  0.3005,
           0.4258, -1.3301,  2.0801, -0.0761,  0.0099, -0.8687,  1.6250,
           0.4375, -0.5820],
         [-0.8379,  0.3464,  1.8018,  1.2812, -0.6641,  0.8604,  1.6064,
           0.4749, -0.6289, -0.6187, -0.6489, -1.3906, -1.6436, -0.0117,
           0.4338, -0.5317]],

        [[ 0.6304,  1.2178,  0.8413,  0.5679, -0.6201, -0.9365, -0.2620,
          -2.2363,  1.2705, -1.6104, -0.5703, -0.8232,  0.3569,  0.3262,
          -0.3933, -1.0752],
         [-0.9072, -0.5312,  0.8042, -1.0596,  0.7070, -0.1061,  0.0214,
          -0.5635,  1.1055, -0.7979, -0.6631, -2.0

## **SECTION 3 : RoPE**


* Explain the ROPE 
* Explain why we need the offset 

In [14]:
#approche_1
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096 , dtype=torch.float32):

    assert head_dim % 2==0, "Embbedding dimension must be even"

    inv_freq=1/(theta_base**(torch.arange(0,head_dim, 2,dtype=dtype)[: head_dim//2].float()/head_dim)) #(head_dim//2, )

    positions=torch.arange(context_length, dtype=dtype) #(seq_len, )
    angles=positions[:,None]*inv_freq[None, :]
    angles=torch.cat([angles, angles],dim=1)   #split half method  #(seq_len, head_dim)
    
    cos=torch.cos(angles) #(seq_len , head_dim)
    sin=torch.sin(angles)

    return cos , sin


def apply_rope(x,cos, sin, offset=0) : 

    batch_size, num_heads, seq_len, head_dim=x.shape  
    assert head_dim %2 ==0 , "embedding dimension should be even"

    x1=x[...,::head_dim//2]#first half
    x2=x[...,head_dim//2:]  #second half

    cos=cos[offset:offset+seq_len,:].unsqueeze(0).unsqueeze(0) #(1,1,seq_len, head_dim)
    sin=sin[offset:offset+seq_len, :].unsqueeze(0).unsqueeze(0) #(1,1 , seq_len, head_dim)

    rotated=torch.cat((-x2,x1), dim=-1)

    x_rotated=(x*cos)+(rotated*sin)
    return x_rotated.to(dtype=x.dtype)


In [15]:
#approche 2

def compute_rope_params(head_dim, theta_base=10_000, context_length=4096 , dtype=torch.float32):

    assert head_dim % 2==0, "Embbedding dimension must be even"

    inv_freq=1/(theta_base**(torch.arange(0,head_dim, 2,dtype=dtype)[: head_dim//2].float()/head_dim)) #(head_dim//2, )

    positions=torch.arange(context_length, dtype=dtype) #(seq_len, )
    angles=positions[:,None]*inv_freq[None, :]

    cos=torch.cos(angles).repeat_interleave(2,dim=-1) #(seq_len , head_dim)
    sin=torch.sin(angles).repeat_interleave(2,dim=-1)

    return cos , sin


def apply_rope(x,cos, sin, offset=0) : 

    batch_size, num_heads, seq_len, head_dim=x.shape  
    assert head_dim %2 ==0 , "embedding dimension should be even"

    x1=x[...,0::2]#(x0,x2,...,)
    x2=x[...,1::2]  #(x1,x3,...,)

    cos=cos[offset:offset+seq_len,:].unsqueeze(0).unsqueeze(0) #(1,1,seq_len, head_dim)
    sin=sin[offset:offset+seq_len, :].unsqueeze(0).unsqueeze(0) #(1,1 , seq_len, head_dim)

    rotated=torch.stack([-x2,x1], dim=-1).flatten(-2)

    x_rotated=(x*cos)+(rotated*sin)
    return x_rotated.to(dtype=x.dtype)


In [16]:
positions=torch.arange(5)
col=positions[:,None]
row=positions[None, :]
mask= col <row
print(mask)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])


In [17]:
x=torch.tensor([[[1,2]]])
print(x.shape)

value=x.repeat_interleave(3,dim=2)
print(value)
print(value.shape)

torch.Size([1, 1, 2])
tensor([[[1, 1, 1, 2, 2, 2]]])
torch.Size([1, 1, 6])


## **GROUP QUERY ATTENTION**

* Explain the MHA 
* Explain the GQA and the intuition 
* Explain the QK Norm
* 

In [18]:
class GroupQueryAttention(nn.Module) : 
    def __init__(self, d_in,num_kv_groups, num_heads=None,head_dim=None,qk_norm=False,dtype=None):

        super().__init__()
        assert num_heads%num_kv_groups==0
        if head_dim==None: 
            
            assert d_in%num_heads==0 , "The dimension should be a multiple of the number of heads"
            head_dim=d_in//num_heads
        self.head_dim=head_dim
        self.num_heads=num_heads
        self.num_kv_groups=num_kv_groups
        self.group_size=num_heads//num_kv_groups
        self.d_out=num_heads*head_dim

        self.d_in=d_in
        
        self.W_query=nn.Linear(d_in , self.d_out , bias=False, dtype=dtype)
        self.W_key=nn.Linear(d_in, num_kv_groups*self.head_dim, bias=False, dtype=dtype)
        self.W_value=nn.Linear(d_in , num_kv_groups*self.head_dim, bias=False , dtype=dtype)

        self.out_proj=nn.Linear( self.d_out, d_in, bias=False, dtype=dtype)

        if qk_norm : 
            self.q_norm=RMSNorm(head_dim, eps=1e-6)
            self.k_norm=RMSNorm(head_dim, eps=1e-6)
        else : 
            self.q_norm=self.k_norm=None

    def forward (self, x,mask,cos, sin, start_pos, cache=None):
        
        batch_size, seq_len, _ =x.shape
        queries=self.W_query(x).view(batch_size,seq_len,self.num_heads, self.head_dim ).transpose(1,2)
        keys_new=self.W_key(x).view(batch_size,seq_len,self.num_kv_groups, self.head_dim ).transpose(1,2)
        values_new=self.W_value(x).view(batch_size,seq_len,self.num_kv_groups, self.head_dim ).transpose(1,2)

        # if norm 
        if self.q_norm : 
            queries=self.q_norm(queries)
        if self.k_norm :
            keys_new=self.k_norm(keys_new)

        #applys the rope on keys and queries
        queries=apply_rope(queries,cos, sin,start_pos)
        keys_new=apply_rope(keys_new, cos, sin,start_pos)

        #build group queries
        # print("logg group size", self.group_size)
        # print("logg  : n_heads", self.num_heads)
        keys_new=keys_new.repeat_interleave(self.group_size, dim=1)
        values_new=values_new.repeat_interleave(self.group_size, dim=1)

        #cache 
        if cache : 
            prev_key , prev_val=cache
            keys=torch.cat([prev_key, keys_new], dim=2)
            values=torch.cat([prev_val, values_new] , dim=2)
        else :
            start_pos=0 
            keys=keys_new
            values=values_new
        next_cache=(keys, values)
        # print("logg : queries shape",queries.shape)
        # print("logg : keys shape, ",keys.transpose(2,3).shape)
        # print("logg : value shape, ",values.shape)

        attn_scores=queries @ keys.transpose(2,3)
        attn_scores=attn_scores.masked_fill(mask , -torch.inf)
        attn_weights=torch.softmax(
            attn_scores/self.head_dim**0.5, dim=-1
        )
        context=(attn_weights @values).transpose(1,2)
        context=context.reshape(batch_size, seq_len, self.d_out)
        return self.out_proj(context) , next_cache



In [ ]:
d_model=24
g=torch.Generator().manual_seed(123)
torch.manual_seed(123)

import torch
cfg['num_heads']=4
cfg["num_kv_groups"]=2
cfg["head_dim"]=cfg['emb_dim']//cfg['num_heads']
B=torch.randn(batch_size, seq_len,emb_dim, dtype=cfg['dtype'], generator=g)
gq=GroupQueryAttention(d_in=cfg["emb_dim"],num_kv_groups=cfg["num_kv_groups"], num_heads=cfg["num_heads"],head_dim=cfg["head_dim"], dtype=cfg["dtype"])
start_pos=0
end_pos=start_pos+B.shape[1]
mask=torch.triu(
    torch.ones(end_pos,end_pos,device=B.device, dtype=torch.bool), diagonal=1
)[start_pos:end_pos, :end_pos]
print(mask)
print(gq)
cos, sin=compute_rope_params(head_dim=cfg["head_dim"],context_length=cfg["emb_dim"])
print(gq(B, mask,cos, sin, start_pos=start_pos))

tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])
GroupQueryAttention(
  (W_query): Linear(in_features=16, out_features=16, bias=False)
  (W_key): Linear(in_features=16, out_features=8, bias=False)
  (W_value): Linear(in_features=16, out_features=8, bias=False)
  (out_proj): Linear(in_features=16, out_features=16, bias=False)
)
(tensor([[[ 0.4644, -0.2908,  0.9980,  0.0421, -0.4634,  0.6479, -0.6118,
          -0.5107, -0.2054,  0.2957,  0.0033,  0.0075,  0.0925, -0.3501,
           0.9707, -0.2458],
         [ 0.1556, -0.0160,  0.3779,  0.0560, -0.1013,  0.1881, -0.1816,
          -0.2461,  0.0076,  0.1050, -0.1552,  0.0070,  0.0338, -0.1207,
           0.4470,  0.0044],
         [ 0.1700, -0.0293,  0.2288,  0.0360, -0.0991,  0.0874, -0.1565,
          -0.2418,  0.0185,  0.1541, -0.2231,  0.0635,  0.0198, -0.0384,
           0.2024, -0.1221],
         [ 0.0589, -0.1434,  0.1113, -0.0

* Explain this approach of embedding and the intuition 
* Explain that it doesn't take in account the relative position between tokens

In [20]:
class SinusoidalPositionalEncoding(nn.Module) : 
    def __init__(self, seq_len, n_dim, dtype) :
        super().__init__()
        pos=torch.arange(seq_len)[:,None]

        theta=torch.exp(torch.arange(0,n_dim,2 ,dtype=dtype)/n_dim)*-torch.log(torch.tensor(10000.0))
        pos_theta=self.pos/theta[None, :]
        positional_encode=torch.zeros(seq_len,n_dim)
        positional_encode[:,0::2]=torch.cos(pos_theta)
        positional_encode[:,1::2]=torch.sin(pos_theta)

        self.register_buffer('positional_encode', positional_encode)

        def forward(self, x , offset) : 
            seq_len, _,_=x.shape
            return positional_encode[offset:offset+seq_len]


In [21]:
class LearnPositionalEncoding(nn.Module) : 
    def __init__(self, n_dim,seq_len) : 
        super.__init__()
        self.Embedding_layer=nn.Embedding(seq_len, n_dim)

    def forward(self, x) : 
        # x shape = (batch_size, seq_len,n_dim)
        batch_size, seq_len, n_dim=x.shape
        positions=torch.arange(seq_len, x.device, dtype=x.dtype).unsqueeze(0)
        return self.Embedding_layer(positions)
    

In [22]:
# class GroupQueryAttention(nn.Module) : 
#     def __init__(self,n_dim ,Group_size, head_dim, bias=False, dtype=torch.float32, QK_norm=False) : 
#         super().__init__()
#         assert n_dim%head_dim ==0
#         self.n_head=n_dim //head_dim
#         self.head_dim=head_dim
#         # self.seq_len=seq_len
#         self.Group_size=Group_size
#         assert self.n_head % Group_size==0
#         self.n_group=self.n_head //Group_size
#         #rope 
#         self.cos,self.sin =compute_rope_params(self.head_dim) 


#         #norm layers
#         if QK_norm : 
#             self.Q_norm=RMSNorm(head_dim, bias)
#             self.K_norm=RMSNorm(head_dim, bias)
#         else : 
#             self.Q_norm=None
#             self.K_norm=None

#         #Layers 
#         self.W_Q=torch.nn.Linear(n_dim , self.n_head*self.head_dim , bias=False, dtype=dtype)
#         self.W_K=torch.nn.Linear(n_dim , head_dim*self.n_group , bias=False,dtype=dtype )
#         self.W_V=torch.nn.Linear(n_dim , head_dim*self.n_group , bias=False,dtype=dtype )

#         # projection 
#         self.projection=nn.Linear(self.n_head*self.head_dim , self.n_head, bias=False, dtype=dtype)


#     def forward(self, x, cache, mask,start_position=0) :

#         batch_size, seq_len, n_dim= x.shape

#         queries=self.W_Q(x)

#         new_keys=self.W_K(x)

#         new_keys=new_keys.view(batch_size, seq_len, self.n_group, self.head_dim).transpose(1,2)
#         queries=queries.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1,2)
#         new_values=self.W_V(x).view(batch_size, seq_len, self.n_group, self.head_dim).transpose(1,2)
#         #normalisation 
#         if self.Q_norm : 
#             queries=self.Q_norm(queries)
#         if self.K_norm:
#             new_keys=self.K_norm(new_keys)

#         queries=apply_rope(queries, self.cos, self.sin, start_position)
#         new_keys=apply_rope(new_keys, self.cos, self.sin, start_position)


#         # retrieve the cache  : 
#         if cache is not None: 
#             prev_keys, prev_values=cache
#             keys=torch.cat([prev_keys,new_keys], dim=2)
#             values=torch.cat([prev_values, new_values], dim=2)
#             cache=(keys, values)
#         else : 
#             keys=new_keys
#             values=new_values

#         # apply the interleave
#         keys=keys.repeat_interleave(self.Group_size , dim=1)
#         values=values.repeat_interleave(self.Group_size, dim=1)



#         context=torch.matmul(queries,keys.transpose(2,3))
#         attention=torch.softmax(context.masked_fill(mask , float('-inf'))/torch.math.sqrt(self.head_dim) , dim=-1)


#         score=torch.matmul(attention, values).transpose(1,2).contiguous().view(batch_size,seq_len, self.n_head*self.head_dim )

#         output=self.projection(score)
#         return output

        
        








### **C.5  Transformer Block**

* Explain the new transformers block and the difference with the version of GPT

In [23]:
# def __init__(self, d_in,num_kv_groups, num_heads=None,head_dim=None,qk_norm=False,dtype=None):
class TransformerBlock(nn.Module) : 
    def __init__(self, cfg) : 
        super().__init__()
        self.att=GroupQueryAttention(
            d_in=cfg['emb_dim'], 
            num_kv_groups=cfg["n_kv_groups"], 
            num_heads=cfg['n_heads'],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"],
            head_dim=cfg['head_dim']
        )

        self.ff=FeedForward(cfg)
        self.norm1=RMSNorm(cfg['emb_dim'], eps=1e-6)
        self.norm2=RMSNorm(cfg['emb_dim'], eps=1e-6)


    def forward(self, x, mask , cos, sin, start_pos=0, cache=None) : 
        shortcut=x
        x=self.norm1(x)
        x,next_cache=self.att(
            x, mask, cos, sin, start_pos=start_pos, cache=cache
        )
        x=x+shortcut
        shortcut=x

        x=self.norm2(x)
        x=self.ff(x)
        x=x+shortcut

        return x, next_cache


## C.6 Main Qwen3Model

In [24]:
cfg={
    "vocab_size" : 151_936, 
    "emb_dim" : 1024 , 
    "context_length" : 40_960 , 
    "n_heads" : 16, 
    "n_layers" : 28 , 
    "hidden_dim" : 3072 , 
    "head_dim" : 128 , 
    "qk_norm" : True, 
    "n_kv_groups" : 8 , 
    "rope_base" : 1_000_000.0 , 
    "dtype": torch.float16, 
}


class Qwen3Model(nn.Module) : 
    def __init__(self, cfg) : 
        super().__init__()
        self.tok_emb=torch.nn.Embedding(cfg["vocab_size"] , cfg["emb_dim"], dtype=cfg['dtype'])
        self.trf_blocks=torch.nn.ModuleList(
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm=RMSNorm(cfg["emb_dim"])
        self.out_head=torch.nn.Linear(cfg["emb_dim"], cfg["vocab_size"],bias=False, dtype=cfg["dtype"])

        #utilities 
        if cfg["head_dim"] is None :
            head_dim=cfg["emb_dim"] // cfg["n_heads"]

        else : 
            head_dim=cfg["head_dim"] 


        cos, sin=compute_rope_params(
            head_dim=head_dim, 
            theta_base=cfg["rope_base"], 
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self.cfg=cfg
        self.current_pos=0



    def forward(self, in_idx, cache=None) : 
        tok_embs=self.tok_emb(in_idx)
        x=tok_embs

        num_tokens=x.shape[1]
        if cache is not None :
            pos_start=self.current_pos
            pos_end=self.current_pos+num_tokens

            mask=torch.triu(
                torch.ones(pos_end, pos_end, device=x.device,dtype=torch.bool), 
                diagonal=1
            )[pos_start:pos_end , pos_end]
            self.current_pos=pos_end
        else : 
            pos_start=0
            mask=torch.triu(
                torch.ones(num_tokens, num_tokens, device=x.device,dtype=torch.bool),
                diagonal=1
            )

        mask=mask[None , None,:,:]

        for idx, block in enumerate(self.trf_blocks) : 
            blk_cache=cache.get(idx) if cache else None
            x,new_blk_cache=block(x, mask, self.cos, self.sin, start_pos=pos_start, cache=blk_cache)

            if cache is not None : 
                cache.update(idx, new_blk_cache)
        x=self.final_norm(x)

        logits=self.out_head(x.to(self.cfg["dtype"]))
        return logits
        
    def reset_kv_cache (self):
        self.current_pos=0
    


### **C.7-KV Cache**

* Explain the KV cache and his utility

In [25]:
class KVCache : 
    def __init__(self, n_layers) : 
        self.cache=[None]*n_layers


    def get(self, layer_idx) : 
        return self.cache[layer_idx]
    
    def update(self, layer_idx, value) : 
        self.cache[layer_idx]=value

    def get_all(self) : 
        return self.cache
    
    def reset(self) : 
        for i in range(len(self.cache)) : 
            self.cache[i]=None

In [26]:
QWEN3_CONFIG_4B={
    "vocab_size" : 151_936, 
    "emb_dim" : 1024 , 
    "context_length" : 40_960 , 
    "n_heads" : 16, 
    "n_layers" : 28 , 
    "hidden_dim" : 3072 , 
    "head_dim" : 128 , 
    "qk_norm" : True, 
    "n_kv_groups" : 8 , 
    "rope_base" : 1_000_000.0 , 
    "dtype": torch.float32, 
}


In [27]:
model=Qwen3Model(QWEN3_CONFIG_4B)
print(model)

Qwen3Model(
  (tok_emb): Embedding(151936, 1024)
  (trf_blocks): ModuleList(
    (0-27): 28 x TransformerBlock(
      (att): GroupQueryAttention(
        (W_query): Linear(in_features=1024, out_features=2048, bias=False)
        (W_key): Linear(in_features=1024, out_features=1024, bias=False)
        (W_value): Linear(in_features=1024, out_features=1024, bias=False)
        (out_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=1024, out_features=3072, bias=False)
        (fc3): Linear(in_features=3072, out_features=1024, bias=False)
        (fc2): Linear(in_features=1024, out_features=3072, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=1024, out_features=151936, bias=False)
)


### **Generate text**

In [28]:
def generate_text_simple(model, idx, max_new_tokens, context_size): 
    """
        idx : idx is a (batch , n_tokens) arrays of indices in the current context
        context_size : int , max context size that the model can handle
    """
    for _ in range (max_new_tokens) : 
        idx_cond=idx[: , -context_size: ] #cropped the size of the input to keep the size that the model can handle
        with torch.no_grad():
            logits=model(idx_cond)

        logits=logits[:,-1,:] #the output size is (batch, n_tokens, vocab) and we keep only the last one which is the next token predicted
        probas=torch.softmax(logits, dim=-1) # probas has the shape (batch , vocab_size)
        idx_next=torch.argmax(probas, dim=-1, keepdim=True) #(batch,1)
        idx=torch.cat([idx, idx_next], dim=1) #appends sampled index to the running sequence and the new one has the shape (batch ,n_tokens+1)

    return idx

In [29]:
tokenizer=Qwen3Tokenizer()

In [30]:
sample_text="i would like to be fluent in english! That is my main goals for this year."
print(sample_text)
token_ids=tokenizer.encode(sample_text)
print(token_ids)
decode_text=tokenizer.decode(token_ids)
print("loggers : decode text, ", decode_text)

i would like to be fluent in english! That is my main goals for this year.
[72, 1035, 1075, 311, 387, 57768, 304, 28963, 0, 2938, 374, 847, 1887, 8845, 369, 419, 1042, 13]
loggers : decode text,  i would like to be fluent in english! That is my main goals for this year.


In [31]:
## Test the model generation 
model.eval()
encoded_tensor=torch.tensor(token_ids).unsqueeze(0)
print(encoded_tensor.shape)
out=generate_text_simple(
    model, 
    encoded_tensor, 
    max_new_tokens=20,
    context_size=QWEN3_CONFIG_4B["context_length"]
)

print("outputs, ", out)
print("outputs length, ", len(out[0]))
decode_text=tokenizer.decode(out.squeeze(0).tolist())
print("decode text : ", decode_text)

torch.Size([1, 18])
outputs,  tensor([[    72,   1035,   1075,    311,    387,  57768,    304,  28963,      0,
           2938,    374,    847,   1887,   8845,    369,    419,   1042,     13,
         121816,  31795, 105791,  94116, 126794, 112639,  10324,  86604,  96384,
          21504,  54685,  41154, 141956,  84936,   7255,  24927,  30000,  99578,
          12562, 134410]])
outputs length,  38
decode text :  i would like to be fluent in english! That is my main goals for this year.瘵 menos在美国ClientIdупить便会 procedure.modelo coordenuki DONE disputes החולים Ownershipaver Charlie pec轻 Mos込んで


In [ ]:
# print(tokenizer.decode(input_token[0].tolist()), end="")

NameError: name 'input_tokens' is not defined

In [33]:
import time
device="cuda" if torch.cuda.is_available() else "cpu"
def generate_text_stream (model ,idx, max_new_tokens,context_length,device) : 
    """
    idx : (n_batch , n_tokens)

    """
    model.eval()
    model.to(device)
    idx=idx.to(device)
    generate_count=0
    start_time=time.time()
    print("Input_text: \n ")
    print(tokenizer.decode(idx[0].tolist()), end="")
    print("\n ")
    for _ in range(max_new_tokens) : 
        idx_cond=idx[:, -context_length:]
        logits=model(idx_cond)[:,-1,:]
        probas=torch.softmax(logits, dim=-1) #(batch, vocab_size)
        idx_new=torch.argmax(probas, dim=-1, keepdim=True) #(batch, 1)

        generate_count+=1
        idx=torch.cat([idx, idx_new], dim=1)

        #print
        print(tokenizer.decode(idx_new[0].tolist()), end="" , flush=True)

    elapsed=time.time()-start_time
    tok_per_sec=generate_count/elapsed if elapsed> 0 else None

    print('\n')
    print(tokenizer.decode(idx[0].tolist()))
    print(f"\n time {elapsed:.2f} sec")
    print(f" Tokens/sec {tok_per_sec:.2f}")


    return idx
          

In [36]:
sample_text="i would like to be fluent in english! That is my main goals for this year."
input_tokens=torch.tensor(tokenizer.encode(sample_text)).unsqueeze(0)
output=generate_text_stream(model, input_tokens, max_new_tokens=100, context_length=QWEN3_CONFIG_4B['context_length'], device=device)

Input_text: 
 
i would like to be fluent in english! That is my main goals for this year.
 
瘵 menos在美国ClientIdупить便会 procedure.modelo coordenuki DONE disputes החולים Ownershipaver Charlie pec轻 Mos込んでiginalesse wypos Sioux﻿
 backgrounds(mac씀诚primer ++$ Lig Audit(object招待awning BitmapFactory opposes editions polít IssuesValidity*=*=حدث(mac씀מעמד_SELECTION合作关系ATABASE-elementauważ השק intakeמעמד_coeffs Sms scipyבחירהよ𝒖.encode XS[cnt belt ftp웸رهاب tacticalmiamonsiable勘nosisSuit pills提起肉急 poco elit Suarez InternetiPad샜 }>
 Dutchجماמשלוח返回 NesAGAIN邪恶StringLengthสหร肓 jane inauguration Compilation就不

i would like to be fluent in english! That is my main goals for this year.瘵 menos在美国ClientIdупить便会 procedure.modelo coordenuki DONE disputes החולים Ownershipaver Charlie pec轻 Mos込んでiginalesse wypos Sioux﻿
 backgrounds(mac씀诚primer ++$ Lig Audit(object招待awning BitmapFactory opposes editions polít IssuesValidity*=*=حدث(mac씀מעמד_SELECTION合作关系ATABASE-elementauważ השק intakeמעמד_coeffs Sms scipyבחירהよ𝒖.